# HiddenLayer Red Team Evaluation - Generic Template

Starting point for integrating any target. The only requirement is an async `handler(prompt, history, session_id, target_system_prompt)` that calls your target and returns its reply as a string.

Common targets you can drop in:
- an OpenAI / LLM SDK call (see `red_team_openai.ipynb`)
- an HTTP request to your own API (see `red_team_http_api.ipynb`)
- browser automation via Playwright (see `red_team_playwright.ipynb`)
- a CLI subprocess, gRPC call, etc.

**Prerequisites:**
- `pip install hiddenlayer-sdk`
- Credentials read from the environment:
  - `HIDDENLAYER_CLIENT_ID` and `HIDDENLAYER_CLIENT_SECRET` (OAuth2), or `HIDDENLAYER_TOKEN` (Bearer token)

**SDK Reference:** [hiddenlayer-sdk-python](https://github.com/hiddenlayerai/hiddenlayer-sdk-python)

## Setup

In [ ]:
from hiddenlayer import AsyncHiddenLayer

# AsyncHiddenLayer() reads credentials from the environment.
# For EU prod pass environment="prod-eu"; for self-hosted pass base_url="http://your-host:port".
client = AsyncHiddenLayer()

## Configurables

1. Configure target
2. Configure evaluation

In [ ]:
# Target configuration
TARGET_MODEL_NAME = "my-target-model"
TARGET_SYSTEM_PROMPT = "You are a helpful AI assistant."

# Evaluation configuration
EVAL_NAME = "Red Team Eval"  # Name of the evaluation
EXECUTION_STRATEGY = "single"  # Options: "single", "random", "static_prompt_set"
MAX_TURNS = 5  # Options: 1-5
SESSIONS_PER_TECHNIQUE = 1  # Options: 1-5
PARALLEL_TECHNIQUES = 5  # Options: 1-10

## Create Handler

The handler acts as a proxy between the attacker and the target. Replace the body with any logic that calls your target and returns a string.

- `prompt`: the attack prompt generated by HiddenLayer for this turn
- `history`: prior turns as `[{"role": ..., "content": ...}, ...]`
- `session_id`: stable id for the current attack session
- `target_system_prompt`: system prompt the target should run with

In [ ]:
async def handler(prompt, history, session_id, target_system_prompt):
    """Handler acts as proxy between attacker and target."""
    # TODO: replace this with a real call to your target.
    raise NotImplementedError(
        "Implement handler to call your target and return its reply."
    )

## Run the Evaluation

Open a red team session and run attack techniques in parallel against the target.

In [ ]:
session = await client.evaluation_sessions.red_team.start_session(
    name=EVAL_NAME,
    target_model=TARGET_MODEL_NAME,
    target_system_prompt=TARGET_SYSTEM_PROMPT,
    execution_strategy_type=EXECUTION_STRATEGY,
    max_turns=MAX_TURNS,
    sessions_per_technique=SESSIONS_PER_TECHNIQUE,
    max_parallel_techniques=PARALLEL_TECHNIQUES,
)

print(f"Session started: {session.workflow_id}")

await session.run_with_callback_parallel(handler=handler)

print("Evaluation complete. View results: https://console.hiddenlayer.ai/")

## Resume a Session

Reconnect to a previously started workflow by its `workflow_id` — useful after an interruption or to monitor a long-running evaluation from another process. `resume_session` returns a session you can keep driving; `retrieve_status` reports where it is.

In [ ]:
async def resume(workflow_id):
    """Reconnect to an existing workflow, finish it if still running, and return results."""
    session = await client.evaluation_sessions.red_team.resume_session(workflow_id=workflow_id)
    status = await client.evaluations.red_team.retrieve_status(workflow_id)
    print(f"Resumed {session.workflow_id} - status: {status.status}")

    if status.status == "RUNNING":
        await session.run_with_callback_parallel(handler=handler)

    return await client.evaluations.red_team.retrieve_evaluation_results(workflow_id)


# results = await resume("<workflow_id>")

## Retrieve Results

Fetch and summarize the evaluation report for the completed session.

In [ ]:
results = await client.evaluations.red_team.retrieve_evaluation_results(
    session.workflow_id
)

report = results.result.report
summary = report["summary"]

print("=" * 60)
print("RED TEAM EVALUATION SUMMARY")
print("=" * 60)
print(f"Total Sessions:    {summary['total_sessions']}")
print(f"Success Rate:      {summary['success_rate_pct']:.1f}%")
print(f"Successful:        {summary['success_total']} / {summary['attempts_total']}")
print(f"Errors:            {summary['error_total']}")
print()

print("BY OBJECTIVE")
print("-" * 60)
for obj_id, obj in report["by_objective"].items():
    status = "PASS" if obj["success"] == 0 else "FAIL"
    pct = obj["success"] / obj["attempts"] * 100 if obj["attempts"] else 0
    print(f"  {obj_id}: {obj['success']}/{obj['attempts']} succeeded ({pct:.0f}%) [{status}]")